In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Instrument calibration: BEER at ESS

This example demonstrates a Rietveld refinement of a duplex steel
structure using time-of-flight neutron powder diffraction data
simulated with McStas.

Two datasets from two symmetrically positioned banks (S2 and N2) of
the BEER instrument are analyzed in this tutorial.

## 🛠️ Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data
from easydiffraction import extract_data_paths_from_zip
from easydiffraction import extract_metadata

## 🧩 Define Structures

This section covers how to add structures and modify their
parameters.

### Create Ferrite Structure

In [3]:
ferrite = StructureFactory.from_scratch(name='ferrite')

ferrite.space_group.name_h_m = 'I m -3 m'
ferrite.space_group.coord_system_code = '1'

ferrite.cell.length_a = 2.886

ferrite.atom_sites.create(
    id='Fe',
    type_symbol='Fe',
    fract_x=0.0,
    fract_y=0.0,
    fract_z=0.0,
    adp_type='Biso',
    adp_iso=1.0,
)

### Create Austenite Structure

In [4]:
austenite = StructureFactory.from_scratch(name='austenite')

austenite.space_group.name_h_m = 'F m -3 m'
austenite.space_group.coord_system_code = '1'

austenite.cell.length_a = 3.6468

austenite.atom_sites.create(
    id='Fe',
    type_symbol='Fe',
    fract_x=0.0,
    fract_y=0.0,
    fract_z=0.0,
    adp_type='Biso',
    adp_iso=1.0,
)

## 🔬 Define Experiments

This section shows how to add experiments, configure their parameters,
and link the structures defined in the previous step.

### Download Data

In [5]:
zip_path = download_data('meas-ferrite-austenite-beer', destination='data')
data_paths = extract_data_paths_from_zip(zip_path, destination='data/calibrate-beer-ess')

data_path_s2 = data_paths[1]  # 'Duplex_in_HR_for_IRF_S2.dat'
data_path_n2 = data_paths[0]  # 'Duplex_in_HR_for_IRF_N2.dat'

Getting data...


Data 'meas-ferrite-austenite-beer': ferrite + austenite, BEER (ESS), S2 and N2 detector bank datasets


✅ Data 'meas-ferrite-austenite-beer' downloaded to '../../../data/meas-ferrite-austenite-beer.zip'


### Create Experiment

In [6]:
expt_s2 = ExperimentFactory.from_data_path(
    name='expt_s2',
    data_path=data_path_s2,
    beam_mode='time-of-flight',
)

In [7]:
expt_n2 = ExperimentFactory.from_data_path(
    name='expt_n2',
    data_path=data_path_n2,
    beam_mode='time-of-flight',
)

### Set Instrument

In [8]:
expt_s2.instrument.setup_twotheta_bank = extract_metadata(
    data_path_s2, r'two_theta\s*=\s*(\d*\.?\d+)'
)
expt_s2.instrument.calib_d_to_tof_linear = extract_metadata(
    data_path_s2, r'DIFC\s*=\s*(\d*\.?\d+)'
)

In [9]:
expt_n2.instrument.setup_twotheta_bank = extract_metadata(
    data_path_n2, r'two_theta\s*=\s*(\d*\.?\d+)'
)
expt_n2.instrument.calib_d_to_tof_linear = extract_metadata(
    data_path_n2, r'DIFC\s*=\s*(\d*\.?\d+)'
)

### Set Peak Profile

In [10]:
expt_s2.peak.show_supported()

Peak types


,,Type,Description
1,,pseudo-voigt,TOF non-convoluted pseudo-Voigt profile
2,*,jorgensen,TOF Jorgensen profile: back-to-back exponentials ⊗ Gaussian
3,,jorgensen-von-dreele,TOF Jorgensen-Von Dreele profile: back-to-back exponentials ⊗ pseudo-Voigt
4,,double-jorgensen-von-dreele,TOF Double-Jorgensen-Von Dreele profile: double back-to-back exponentials ⊗ pseudo-Voigt (Z-Rietveld type0m)


In [11]:
expt_s2.peak.type = 'pseudo-voigt'

⚠️ Switching peak profile type removes these settings:                                                                            
   • decay_beta_0                                                                                                                 
   • decay_beta_1                                                                                                                 
   • rise_alpha_0                                                                                                                 
   • rise_alpha_1                                                                                                                 


⚠️ Switching peak profile type adds these settings with defaults:                                                                 
   • broad_lorentz_gamma_0=0.0                                                                                                    
   • broad_lorentz_gamma_1=0.0                                                                                                    
   • broad_lorentz_gamma_2=0.0                                                                                                    
   • broad_lorentz_size_l=0.0                                                                                                     
   • broad_lorentz_strain_l=0.0                                                                                                   


Peak profile type for experiment 'expt_s2' changed to


pseudo-voigt


In [12]:
expt_s2.peak.broad_gauss_sigma_0 = 300
expt_s2.peak.broad_gauss_sigma_1 = 1200
expt_s2.peak.broad_gauss_sigma_2 = 900

In [13]:
expt_n2.peak.type = 'pseudo-voigt'

⚠️ Switching peak profile type removes these settings:                                                                            
   • decay_beta_0                                                                                                                 
   • decay_beta_1                                                                                                                 
   • rise_alpha_0                                                                                                                 
   • rise_alpha_1                                                                                                                 


⚠️ Switching peak profile type adds these settings with defaults:                                                                 
   • broad_lorentz_gamma_0=0.0                                                                                                    
   • broad_lorentz_gamma_1=0.0                                                                                                    
   • broad_lorentz_gamma_2=0.0                                                                                                    
   • broad_lorentz_size_l=0.0                                                                                                     
   • broad_lorentz_strain_l=0.0                                                                                                   


Peak profile type for experiment 'expt_n2' changed to


pseudo-voigt


In [14]:
expt_n2.peak.broad_gauss_sigma_0 = 300
expt_n2.peak.broad_gauss_sigma_1 = 1200
expt_n2.peak.broad_gauss_sigma_2 = 900

### Set Background

In [15]:
expt_s2.background.show_supported()

Background types


,,Type,Description
1,,chebyshev,Chebyshev polynomial background
2,*,line-segment,Linear interpolation between points


In [16]:
expt_s2.background.auto_estimate()

In [17]:
expt_s2.background.show()

Line-segment background points


,Position,Intensity
1,40094.52340,0.00154
2,48089.84890,0.16817
3,51976.02300,0.18318
4,54301.35670,0.17817
5,58729.04700,0.29095
6,62710.78280,0.30387
7,67711.84290,0.25310
8,73158.85750,0.22573
9,76439.80780,0.20324
10,81249.74460,0.19453


In [18]:
for point in expt_s2.background:
    expt_n2.background.create(
        id=point.id.value, position=point.position.value, intensity=point.intensity.value
    )

### Set Linked Structures

In [19]:
expt_s2.linked_structures.create(structure_id='ferrite', scale=285)
expt_s2.linked_structures.create(structure_id='austenite', scale=68)

In [20]:
expt_n2.linked_structures.create(structure_id='ferrite', scale=285)
expt_n2.linked_structures.create(structure_id='austenite', scale=68)

### Set Excluded Regions

In [21]:
expt_s2.excluded_regions.create(id='1', start=0, end=40500)
expt_s2.excluded_regions.create(id='2', start=130000, end=180000)

In [22]:
expt_n2.excluded_regions.create(id='1', start=0, end=40500)
expt_n2.excluded_regions.create(id='2', start=130000, end=180000)

## 📦 Define Project

The project object is used to manage the structure, experiments,
and analysis

### Create Project

In [23]:
project = Project(name='beer_mcstas')
project.save_as(dir_path='projects/calibrate-beer-ess')

Saving project 📦 'beer_mcstas' to '../../../projects/calibrate-beer-ess'


├── 📄 project.edi


├── 📁 structures/


├── 📁 experiments/


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 beer_mcstas.html


### Add Structures

In [24]:
project.structures.add(ferrite)
project.structures.add(austenite)

### Add Experiments

In [25]:
project.experiments.add(expt_s2)
project.experiments.add(expt_n2)

### Display Structure

In [26]:
project.display.structure(struct_name='ferrite')
project.display.structure(struct_name='austenite')

Structure 🧩 'ferrite' (Atom view type: 'covalent')


Structure 🧩 'austenite' (Atom view type: 'covalent')


### Display Pattern

In [27]:
project.display.pattern(expt_name='expt_s2')

In [28]:
project.display.pattern(expt_name='expt_n2')

## 🚀 Perform Analysis

This section shows the analysis process, including how to set up
calculation and fitting engines.

### Set Fit Mode

In [29]:
project.analysis.fitting_mode.show_supported()

Fitting Mode types


,,Type,Description
1,,joint,Fit several experiments together with shared parameters.


In [30]:
project.analysis.fitting_mode.type = 'joint'

Fitting mode changed to


joint


### Set Free Parameters

In [31]:
project.display.parameters.fittable()

Fittable parameters for all structures (🧩 data blocks)


,datablock,category,entry,parameter,value,uncertainty,units,free
1,ferrite,cell,,length_a,2.88600,,Å,False
2,ferrite,atom_site,Fe,occupancy,1.00000,,,False
3,ferrite,atom_site,Fe,adp_iso,1.00000,,Å²,False
4,austenite,cell,,length_a,3.64680,,Å,False
5,austenite,atom_site,Fe,occupancy,1.00000,,,False
6,austenite,atom_site,Fe,adp_iso,1.00000,,Å²,False


Fittable parameters for all experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,units,free
1,expt_s2,linked_structure,ferrite,scale,285.00000,,,False
2,expt_s2,linked_structure,austenite,scale,68.00000,,,False
3,expt_s2,peak,,broad_lorentz_gamma_0,0.00000,,μs,False
4,expt_s2,peak,,broad_lorentz_gamma_1,0.00000,,μs/Å,False
5,expt_s2,peak,,broad_lorentz_gamma_2,0.00000,,μs²/Å²,False
6,expt_s2,peak,,broad_lorentz_size_l,0.00000,,μs²/Å²,False
7,expt_s2,peak,,broad_lorentz_strain_l,0.00000,,μs/Å,False
8,expt_s2,peak,,broad_gauss_sigma_0,300.00000,,μs²,False
9,expt_s2,peak,,broad_gauss_sigma_1,1200.00000,,μs/Å,False
10,expt_s2,peak,,broad_gauss_sigma_2,900.00000,,μs²/Å²,False


In [32]:
ferrite.atom_sites['Fe'].adp_iso.free = True
austenite.atom_sites['Fe'].adp_iso.free = True

In [33]:
expt_s2.linked_structures['ferrite'].scale.free = True
expt_s2.linked_structures['austenite'].scale.free = True

expt_s2.peak.broad_gauss_sigma_0.free = True
expt_s2.peak.broad_gauss_sigma_1.free = True
expt_s2.peak.broad_gauss_sigma_2.free = True
expt_s2.peak.broad_lorentz_gamma_0.free = True

expt_s2.instrument.calib_d_to_tof_offset.free = True

for segment in expt_s2.background:
    segment.intensity.free = True

In [34]:
expt_n2.linked_structures['ferrite'].scale.free = True
expt_n2.linked_structures['austenite'].scale.free = True

expt_n2.peak.broad_gauss_sigma_0.free = True
expt_n2.peak.broad_gauss_sigma_1.free = True
expt_n2.peak.broad_gauss_sigma_2.free = True
expt_n2.peak.broad_lorentz_gamma_0.free = True

expt_n2.instrument.calib_d_to_tof_offset.free = True

for segment in expt_n2.background:
    segment.intensity.free = True

### Add Constraints

In [35]:
project.analysis.aliases.create(
    id='s2_ferrite_scale', param=expt_s2.linked_structures['ferrite'].scale
)
project.analysis.aliases.create(
    id='s2_austenite_scale', param=expt_s2.linked_structures['austenite'].scale
)

project.analysis.aliases.create(
    id='n2_ferrite_scale', param=expt_n2.linked_structures['ferrite'].scale
)
project.analysis.aliases.create(
    id='n2_austenite_scale', param=expt_n2.linked_structures['austenite'].scale
)

project.analysis.constraints.create(expression='n2_ferrite_scale = s2_ferrite_scale')
project.analysis.constraints.create(expression='n2_austenite_scale = s2_austenite_scale')

### Run Fitting

Run full fitting with all free parameters.

In [36]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Using all experiments 🔬 ['expt_s2', 'expt_n2'] for 'joint' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.06,350.70,
2,58,3.39,34.88,90.1% ↓
3,113,6.37,12.86,63.1% ↓
4,205,11.38,12.85,
5,293,16.55,12.85,
6,334,24.79,12.85,


...

│   └── 📄 expt_n2.edi


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 beer_mcstas.html


Fix background and run fitting again.

In [37]:
for segment in expt_s2.background:
    segment.intensity.free = False
for segment in expt_n2.background:
    segment.intensity.free = False

In [38]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Using all experiments 🔬 ['expt_s2', 'expt_n2'] for 'joint' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.09,12.76,
2,19,1.84,12.76,


...

│   └── 📄 expt_n2.edi


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 beer_mcstas.html


Show fit results and parameter correlations.

In [39]:
project.display.fit.results()
project.display.fit.correlations()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),1.84
4,🔁 Iterations,16
5,📏 Goodness-of-fit (reduced χ²),12.76
6,"📏 R-factor (Rf, %)",4.97
7,"📏 R-factor squared (Rf², %)",3.89
8,"📏 Weighted R-factor (wR, %)",3.40


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,ferrite,atom_site,Fe,adp_iso,Å²,1.7191,1.7191,0.0126,0.00 % ↑
2,austenite,atom_site,Fe,adp_iso,Å²,1.7126,1.7126,0.0118,0.00 % ↑
3,expt_s2,linked_structure,ferrite,scale,,402.6550,402.6552,1.1814,0.00 % ↑
4,expt_s2,linked_structure,austenite,scale,,96.0438,96.0439,0.2381,0.00 % ↑
5,expt_s2,peak,,broad_lorentz_gamma_0,μs,5.2351,5.2351,0.1470,0.00 % ↑
6,expt_s2,peak,,broad_gauss_sigma_0,μs²,884.3508,884.3547,56.2994,0.00 % ↑
7,expt_s2,peak,,broad_gauss_sigma_1,μs/Å,1284.1469,1284.1433,56.2421,0.00 % ↓
8,expt_s2,peak,,broad_gauss_sigma_2,μs²/Å²,311.1744,311.1751,11.1945,0.00 % ↑
9,expt_s2,instrument,,d_to_tof_offset,μs,-10.4534,-10.4534,0.1176,0.00 % ↑
10,expt_n2,peak,,broad_lorentz_gamma_0,μs,5.2429,5.2429,0.1475,0.00 % ↑


### Display Pattern

Show full range in TOF.

In [40]:
project.display.pattern(expt_name='expt_s2')

In [41]:
project.display.pattern(expt_name='expt_n2')

Show selected peaks in d-spacing.

In [42]:
project.display.pattern(
    expt_name='expt_s2',
    x='d_spacing',
    x_min=2.08,
    x_max=2.13,
)

In [43]:
project.display.pattern(
    expt_name='expt_n2',
    x='d_spacing',
    x_min=2.08,
    x_max=2.13,
)